In [1]:
# [셀 1: 🛠️ 글로벌 설정 및 라이브러리 임포트 (수정됨 - 앙상블 적용)]

import os
import pandas as pd
import glob
import json
import math
import subprocess
import time
import re
from datetime import datetime
import numpy as np 

# --- RDKit 모듈 임포트 ---
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, Lipinski, QED
    from rdkit import RDConfig
    from rdkit.Chem import FilterCatalog 
    
    import sys
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    try:
        import sascorer
    except ImportError:
        print("⚠️ [경고] SA_Score 모듈을 찾을 수 없습니다. SA 점수는 0으로 처리됩니다.")
        sascorer = None

    print("✅ RDKit 및 관련 모듈 로드 시도 완료.")
except ImportError:
    print("❌ [오류] RDKit 모듈을 찾을 수 없습니다. 'drugai' 환경에서 실행 중인지 확인하세요.")

# --- (1) 환경 이름 및 경로 ---
POCKET2MOL_ENV_NAME = "Pocket2Mol"
POCKET2MOL_DIR = "/home/nudge/Pocket2Mol"
GNINA_ENV_NAME = "gnina"

# --- (2) drugai 프로젝트 경로 ---
DRUGAI_DIR = "/home/nudge/Project/drugai"

# --- (3) ★★★ 모델 경로 설정 (앙상블용 수정) ★★★ ---
POCKET2MOL_CONFIG = "/home/nudge/Pocket2Mol/configs/sample_for_pdb.yml"
ADMET_MODEL_CKPT_DIR = "/home/nudge/Project/drugai/models"
GNINA_EXECUTABLE_PATH = "/home/nudge/gnina_binary" 

# [수정] 단일 체크포인트 대신 3개 모델 리스트 정의
AFFINITY_CKPT_DIR = "/home/nudge/Project/drugai/checkpoints"
AFFINITY_ENSEMBLE_MODELS = [
    "egnn_seed41.pt", 
    "egnn_seed42.pt", 
    "egnn_seed43.pt"
]
print(f"✅ Binding Affinity 모델: 앙상블 적용 ({len(AFFINITY_ENSEMBLE_MODELS)}개 모델)")

# --- (4) 고정 타겟 정보 (분자 생성용) ---
GENERATION_TARGET = {
    "name": "TTR_V30M",
    "pdb_path": "/home/nudge/Project/drugai/data/PDB/input/receptor/TTR_V30M.pdb",
    "center": [42.63, 42.89, 89.61],
    "bbox_size": 20.0
}

# --- (5) 고정 타겟 정보 (Gnina 도킹용) ---
RECEPTORS_CONFIG = [
    {
        "name": "MT_V30M", 
        "pdb_path": "/home/nudge/Project/drugai/data/PDB/input/receptor/TTR_V30M.pdb",
        "center": [42.63, 42.89, 89.61],
        "size": 20.0
    },
    {
        "name": "TTR_WT",
        "pdb_path": "/home/nudge/Project/drugai/data/PDB/input/receptor/TTR.pdb",
        "center": [21.24, 43.08, 4.85],
        "size": 20.0
    }
]

# --- (6) 고정 대조군 정보 ---
CONTROL_LIGANDS_SMILES = [
    {"id": "LIG_Aspirin", "SMILES": "CC(=O)Oc1ccccc1C(=O)O"}
]
CONTROL_LIGANDS_PDB = [
    {"id": "LIG_T44", "path": "/home/nudge/Project/drugai/data/PDB/input/ligand/TT4.pdb"},
    {"id": "LIG_Tafamidis", "path": "/home/nudge/Project/drugai/data/PDB/input/ligand/tifamidis.pdb"}
]

# --- (7) 기타 설정 ---
DEVICE = "cuda:0"
NUM_GENERATIONS_PER_CYCLE = 2000
TAFAMIDIS_SMILES = "O=C(O)c1ccc2nc(oc2c1)c1cc(cc(c1)Cl)Cl"

# --- (8) 결과 저장 폴더 생성 ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = f"./pipeline_results_{timestamp}"
os.makedirs(RESULTS_DIR, exist_ok=True)
smiles_output_dir = os.path.abspath(os.path.join(RESULTS_DIR, "1_generated_molecules"))
os.makedirs(smiles_output_dir, exist_ok=True)

print("\n✅ [셀 1] 글로벌 설정 완료.")

✅ RDKit 및 관련 모듈 로드 시도 완료.
✅ Binding Affinity 모델: 앙상블 적용 (3개 모델)

✅ [셀 1] 글로벌 설정 완료.


In [2]:
# [셀 2: 1단계 Pocket2Mol 분자 생성 (수정됨)]

import time
import glob 

print("[1단계 실행]... Pocket2Mol 분자 생성을 시작합니다.")
print(f"(대상 환경: {POCKET2MOL_ENV_NAME})")

script_path = os.path.join(POCKET2MOL_DIR, "sample_for_pdb.py") 

# --- 🔻 [수정] GENERATION_TARGET 딕셔너리 사용 ---
center_str = f"\"{','.join(map(str, GENERATION_TARGET['center']))}\""
bbox_str = str(GENERATION_TARGET['bbox_size']) 
pdb_file = GENERATION_TARGET['pdb_path']
# --- 🔺 ---------------------------------------

cmd1 = (
    f"conda run -n {POCKET2MOL_ENV_NAME} python {script_path} "
    f"--config {POCKET2MOL_CONFIG} "
    f"--pdb_path {pdb_file} " 
    f"--center {center_str} "
    f"--bbox_size {bbox_str} "
    f"--outdir {smiles_output_dir} " 
    f"--device {DEVICE}"
)

print(f"CMD: {cmd1}")
!{cmd1}

print("\n[INFO] Pocket2Mol 실행 완료. 파일 저장을 위해 3초 대기합니다...")
time.sleep(3) 

print(f"\n[INFO] '{smiles_output_dir}' 폴더 내부에서 'SMILES.txt' 파일을 검색합니다...")

# 글로벌 변수 선언
global smiles_file_path 

search_pattern = os.path.join(smiles_output_dir, "**", "SMILES.txt")
found_files = glob.glob(search_pattern, recursive=True)

if not found_files:
    print(f"❌ [1단계 오류] '{smiles_output_dir}' 하위 폴더에서 'SMILES.txt' 파일을 찾을 수 없습니다.")
    smiles_file_path = "/tmp/SMILES_NOT_FOUND.txt"
else:
    smiles_file_path = max(found_files, key=os.path.getmtime)
    print(f"\n[1단계 완료] SMILES 파일 자동 찾기 성공!")
    print(f"   > 찾은 경로: {smiles_file_path}")
    !head -n 5 {smiles_file_path}

[1단계 실행]... Pocket2Mol 분자 생성을 시작합니다.
(대상 환경: Pocket2Mol)
CMD: conda run -n Pocket2Mol python /home/nudge/Pocket2Mol/sample_for_pdb.py --config /home/nudge/Pocket2Mol/configs/sample_for_pdb.yml --pdb_path /home/nudge/Project/drugai/data/PDB/input/receptor/TTR_V30M.pdb --center "42.63,42.89,89.61" --bbox_size 20.0 --outdir /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/1_generated_molecules --device cuda:0
tensor([8.6044, 0.8471])
tensor([8.8073, 1.2883])
tensor([5.8667, 1.3647, 0.2216])
tensor([8.5591, 1.2524])
tensor([5.6712, 1.1426, 1.3321, 0.1294])
tensor([5.8276, 1.3634, 0.2775, 1.4349])
tensor([7.9496, 1.3463, 0.2090, 1.4188, 1.4440])
tensor([5.8584, 1.4548, 0.3525, 1.4853, 1.3027])
tensor([1.3143, 3.4544, 0.2657, 1.4462, 1.4554])
tensor([7.9304, 0.2657, 1.4462, 1.4554])
tensor([0.8899, 6.2428, 5.8838, 1.4413, 0.2476])
tensor([5.2208, 0.8899, 6.2428, 1.4413, 0.2476])
tensor([0.9437, 6.4335, 1.4698, 0.2154])
tensor([6.3179, 0.9437, 1.4698, 0.2154])
tensor([4.2410,

In [42]:
# [수정된 셀 2.1: Step 1 - 🧪 PhysChem + QED + SA 필터링]

import os
import sys
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED, RDConfig

# --- SA Score 모듈 경로 설정 (RDKit Contrib 폴더) ---
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
try:
    import sascorer
    print("✅ SA_Score 모듈 로드 성공.")
except ImportError:
    sascorer = None
    print("⚠️ [경고] SA_Score 모듈을 찾을 수 없습니다. (SA 필터링은 건너뜁니다.)")

print("[Step 1 실행]... 기초 물성(PhysChem) + QED/SA 필터링을 시작합니다.")

# --- 1. 필터링 기준 설정 (업데이트됨) ---
CRITERIA = {
    "MW":       500.0,  # < 500
    "LogP":     5.0,    # < 5
    "HBD":      5,      # < 5
    "HBA":      10,     # < 10
    "TPSA":     140.0,  # < 140 (투과도 마지노선)
    "RotBonds": 10,     # < 10 (유연성 제한)
    "QED":      0.5,    # > 0.5 (최소 약물성 점수)  ◀ [추가]
    "SA":       6.0     # < 6.0 (합성 난이도 상한선) ◀ [추가]
}

print(f"   > 필터 기준: {CRITERIA}")

# --- 2. 파일 경로 설정 ---
# 안전장치: 경로 변수가 없으면 다시 찾기
if 'smiles_file_path' not in globals():
    import glob
    search_pattern = os.path.join(smiles_output_dir, "**", "SMILES.txt")
    found_files = glob.glob(search_pattern, recursive=True)
    if found_files:
        smiles_file_path = max(found_files, key=os.path.getmtime)
    else:
        smiles_file_path = "SMILES_NOT_FOUND" # 에러 유도

input_smiles_file = smiles_file_path
output_filtered_path = os.path.abspath(os.path.join(RESULTS_DIR, "SMILES_step1_physchem_passed.txt"))

# --- 3. 필터링 함수 정의 ---
def check_drug_likeness(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, "Invalid Structure"
    
    try:
        # 1. 기본 물성 계산
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        rot_bonds = Lipinski.NumRotatableBonds(mol)
        
        # 2. QED 계산
        qed_val = QED.qed(mol)
        
        # 3. SA Score 계산 (모듈 없으면 1.0으로 가정하여 통과)
        if sascorer:
            try: sa_val = sascorer.calculateScore(mol)
            except: sa_val = 10.0 # 계산 에러나면 탈락시키기 위해 높은 점수 부여
        else:
            sa_val = 1.0 

        # --- 판정 (Fail 조건) ---
        if mw > CRITERIA["MW"]: return False, "MW Fail"
        if logp > CRITERIA["LogP"]: return False, "LogP Fail"
        if hbd > CRITERIA["HBD"]: return False, "HBD Fail"
        if hba > CRITERIA["HBA"]: return False, "HBA Fail"
        if tpsa > CRITERIA["TPSA"]: return False, "TPSA Fail"
        if rot_bonds > CRITERIA["RotBonds"]: return False, "RotBonds Fail"
        
        # [추가된 필터]
        if qed_val < CRITERIA["QED"]: return False, "QED Low"  # 0.5 미만 탈락
        if sa_val > CRITERIA["SA"]: return False, "SA High"    # 6.0 초과 탈락
        
        return True, "Pass"
    
    except Exception as e:
        return False, f"Calc Error"

# --- 4. 실행 ---
passed_list = []
fail_counts = {
    "Invalid Structure":0, "Calc Error":0,
    "MW Fail":0, "LogP Fail":0, "HBD Fail":0, "HBA Fail":0, 
    "TPSA Fail":0, "RotBonds Fail":0, 
    "QED Low":0, "SA High":0  # 카운터 추가
}

# 인코딩 처리
try:
    with open(input_smiles_file, 'r', encoding='utf-8') as f: lines = f.readlines()
except:
    with open(input_smiles_file, 'r') as f: lines = f.readlines()

print(f"   > 총 {len(lines)}개 분자 검사 시작...")

for line in lines:
    parts = line.strip().split(',')
    smiles = parts[0]
    mol_id = parts[1] if len(parts) > 1 else f"MOL_{len(passed_list)+1}"
    
    is_pass, reason = check_drug_likeness(smiles)
    
    if is_pass:
        passed_list.append(f"{smiles},{mol_id}")
    else:
        fail_counts[reason] = fail_counts.get(reason, 0) + 1

# --- 5. 결과 저장 ---
with open(output_filtered_path, 'w') as f:
    for item in passed_list:
        f.write(item + "\n")

# --- 6. 리포트 출력 ---
print("-" * 40)
print(f"✅ Step 1 필터링 완료 (QED/SA 포함)!")
print(f"   - 입력: {len(lines)}개")
print(f"   - 통과: {len(passed_list)}개 (생존율: {len(passed_list)/len(lines)*100:.1f}%)")
print(f"   - 탈락 사유별 통계:")
for reason, count in fail_counts.items():
    if count > 0:
        print(f"     • {reason}: {count}개")
print("-" * 40)
print(f"   💾 저장됨: {output_filtered_path}")

# 경로 업데이트
smiles_file_path = output_filtered_path

✅ SA_Score 모듈 로드 성공.
[Step 1 실행]... 기초 물성(PhysChem) + QED/SA 필터링을 시작합니다.
   > 필터 기준: {'MW': 500.0, 'LogP': 5.0, 'HBD': 5, 'HBA': 10, 'TPSA': 140.0, 'RotBonds': 10, 'QED': 0.5, 'SA': 6.0}
   > 총 1270개 분자 검사 시작...
----------------------------------------
✅ Step 1 필터링 완료 (QED/SA 포함)!
   - 입력: 1270개
   - 통과: 1021개 (생존율: 80.4%)
   - 탈락 사유별 통계:
     • QED Low: 249개
----------------------------------------
   💾 저장됨: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_step1_physchem_passed.txt


In [43]:
# [새 셀: Step 2 - 🛑 MedChem 구조 필터링 (PAINS & Brenk)]

import os
from rdkit import Chem
from rdkit.Chem import FilterCatalog

print("[Step 2 실행]... 의약화학적 구조(MedChem) 필터링을 시작합니다.")

# --- 1. 필터 카탈로그 설정 (PAINS + Brenk) ---
params = FilterCatalog.FilterCatalogParams()

# PAINS (가짜 양성 제거)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
# Brenk (독성 및 불안정 구조 제거)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
# NIH (추가적인 원하지 않는 구조 - 선택사항, 여기선 Brenk와 많이 겹치므로 생략 가능하나 포함하면 더 엄격함)
# params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.NIH)

catalog = FilterCatalog.FilterCatalog(params)

print(f"   > 적용된 필터: PAINS, Brenk (총 {catalog.GetNumEntries()}개 규칙)")

# --- 2. 파일 경로 설정 ---
# 이전 단계(Step 1)에서 필터링된 파일을 입력으로 받습니다.
if 'smiles_file_path' not in globals():
     # 변수가 끊겼을 경우를 대비한 안전장치 (직전 단계 파일명 지정)
    input_smiles_file = os.path.abspath(os.path.join(RESULTS_DIR, "SMILES_step1_physchem_passed.txt"))
else:
    input_smiles_file = smiles_file_path

output_medchem_path = os.path.abspath(os.path.join(RESULTS_DIR, "SMILES_step2_medchem_passed.txt"))

# --- 3. 필터링 실행 ---
passed_list = []
fail_counts = {"PAINS": 0, "Brenk": 0, "Error": 0}

# 파일 읽기
try:
    with open(input_smiles_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
except:
    with open(input_smiles_file, 'r') as f:
        lines = f.readlines()

print(f"   > 총 {len(lines)}개 분자 구조 검사 시작...")

for line in lines:
    parts = line.strip().split(',')
    smiles = parts[0]
    mol_id = parts[1] if len(parts) > 1 else f"MOL_Unknown"
    
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        fail_counts["Error"] += 1
        continue

    # 필터 검사
    if catalog.HasMatch(mol):
        # 어떤 필터에 걸렸는지 확인 (첫 번째 매칭만 확인)
        entry = catalog.GetFirstMatch(mol)
        desc = entry.GetDescription()
        
        if "PAINS" in desc.upper():
            fail_counts["PAINS"] += 1
        else:
            fail_counts["Brenk"] += 1 # Brenk 혹은 기타
    else:
        # 통과
        passed_list.append(line.strip())

# --- 4. 결과 저장 ---
with open(output_medchem_path, 'w') as f:
    for item in passed_list:
        f.write(item + "\n")

# --- 5. 리포트 출력 ---
print("-" * 40)
print(f"✅ Step 2 필터링 완료!")
print(f"   - 입력: {len(lines)}개")
print(f"   - 통과: {len(passed_list)}개 (생존율: {len(passed_list)/len(lines)*100:.1f}%)")
print(f"   - 탈락 사유별 통계:")
print(f"     • PAINS (가짜 양성): {fail_counts['PAINS']}개")
print(f"     • Brenk (독성/불안정): {fail_counts['Brenk']}개")
print(f"     • 구조 오류: {fail_counts['Error']}개")
print("-" * 40)
print(f"   💾 저장됨: {output_medchem_path}")

# [중요] 변수 업데이트 (다음 단계인 ADMET 예측으로 넘겨주기 위함)
smiles_file_path = output_medchem_path
print(f"   🔗 다음 단계 입력 파일 경로 업데이트 완료: {smiles_file_path}")

[Step 2 실행]... 의약화학적 구조(MedChem) 필터링을 시작합니다.
   > 적용된 필터: PAINS, Brenk (총 585개 규칙)
   > 총 1021개 분자 구조 검사 시작...
----------------------------------------
✅ Step 2 필터링 완료!
   - 입력: 1021개
   - 통과: 1021개 (생존율: 100.0%)
   - 탈락 사유별 통계:
     • PAINS (가짜 양성): 0개
     • Brenk (독성/불안정): 0개
     • 구조 오류: 0개
----------------------------------------
   💾 저장됨: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_step2_medchem_passed.txt
   🔗 다음 단계 입력 파일 경로 업데이트 완료: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_step2_medchem_passed.txt


In [44]:
# [새 셀 2.5: ADMET 예측 전 대조군 추가]

import shutil

print("[2.5단계 실행]... ADMET 예측을 위해 원본 SMILES 파일에 대조군을 추가합니다.")

# --- ★★★ 대조군 리간드 정의 (셀 3.6에서 가져옴) ★★★ ---
control_ligands_data = [
    ("OC(=O)c1ccc2nc(oc2c1)c3cc(Cl)cc(Cl)c3", "LIG_Tafamidis"),
    ("N[CH](Cc1cc(I)c(Oc2cc(I)c(O)c(I)c2)c(I)c1)C(O)=O", "LIG_T44"),
    ("CC(=O)Oc1ccccc1C(=O)O", "LIG_Aspirin")
]
# -----------------------------------

# [셀 2]에서 생성된 원본 SMILES 파일 경로 (e.g., .../1_generated_molecules/SMILES.txt)
original_smiles_file = smiles_file_path 

# 대조군이 추가될 새 파일 경로
smiles_with_controls_path = os.path.abspath(os.path.join(RESULTS_DIR, "SMILES_with_controls.txt"))

try:
    # 1. 원본 파일 (mol_...)을 새 파일로 복사
    if not os.path.exists(original_smiles_file):
        raise FileNotFoundError(f"원본 SMILES 파일({original_smiles_file})을 찾을 수 없습니다. [셀 2]를 먼저 실행하세요.")
        
    shutil.copyfile(original_smiles_file, smiles_with_controls_path)

    # 2. 복사된 파일(smiles_with_controls_path)에 대조군 '추가(a)'
    with open(smiles_with_controls_path, 'a', encoding='utf-8') as f:
        f.write("\n") # 기존 파일 끝에서 줄바꿈
        for smiles, lig_id in control_ligands_data:
            # 'SMILES,ID' 형식으로 저장
            f.write(f"{smiles},{lig_id}\n")
            
    print(f"✅ [2.5단계 완료] 대조군 포함 SMILES 저장: {smiles_with_controls_path}")
    print(f"   (원본 {original_smiles_file}은 변경되지 않았습니다.)")
    print("\n--- 파일 끝 5줄 (대조군 확인) ---")
    !tail -n 5 {smiles_with_controls_path}

except Exception as e:
    print(f"❌ [2.5단계 오류] 대조군 추가 실패: {e}")

[2.5단계 실행]... ADMET 예측을 위해 원본 SMILES 파일에 대조군을 추가합니다.
✅ [2.5단계 완료] 대조군 포함 SMILES 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_with_controls.txt
   (원본 /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_step2_medchem_passed.txt은 변경되지 않았습니다.)

--- 파일 끝 5줄 (대조군 확인) ---
COC1=CC2=C3C=Cc4c5n(c(=O)n4C3CCC2=CC1)C(=O)CN(C(=O)C1=Cc2ccc(C)cc2C1)C5,MOL_1889

OC(=O)c1ccc2nc(oc2c1)c3cc(Cl)cc(Cl)c3,LIG_Tafamidis
N[CH](Cc1cc(I)c(Oc2cc(I)c(O)c(I)c2)c(I)c1)C(O)=O,LIG_T44
CC(=O)Oc1ccccc1C(=O)O,LIG_Aspirin


In [45]:
# [셀 3: 3단계 ADMET 예측 (재실행)]

import os

# 이 셀은 'drugai' 커널에서 직접 실행됩니다.
print("[3단계 실행]... ADMET 예측을 시작합니다. (대조군 포함, 모델 파일 지정)")

# [수정] 누락된 변수 정의 (RESULTS_DIR은 셀 1에서 정의됨)
admet_csv_path = os.path.abspath(os.path.join(RESULTS_DIR, "3_admet_results.csv"))

# [사용자 지정] ADMET 모델 파일명 (경로 제외, 파일명만)
MODEL_FILES = [
    "admet_hERG_optimized_complete.pt",
    "admet_CYP3A4_optimized_complete.pt",
    "admet_AqSolDB_optimized_complete.pt"
]

# 모델 파일들이 실제로 존재하는지 먼저 검증
missing_files = []
for fname in MODEL_FILES:
    full_path = os.path.join(ADMET_MODEL_CKPT_DIR, fname)
    if not os.path.exists(full_path):
        missing_files.append(fname)

if missing_files:
    print("❌ [설정 오류] 다음 모델 파일을 찾을 수 없습니다. 경로를 확인하세요!")
    for f in missing_files:
        print(f"   - {os.path.join(ADMET_MODEL_CKPT_DIR, f)}")
    raise FileNotFoundError("필수 모델 파일이 없습니다.")

# 파일명들을 공백으로 연결하여 문자열 생성
model_files_str = " ".join(MODEL_FILES)

module_path = "src.admet_multi.predict_admet_gnn" 

# [명령어 구성]
cmd3 = (
    f"PYTHONPATH={DRUGAI_DIR} "
    f"python -m {module_path} "
    f"--smiles_file {smiles_with_controls_path} "
    f"--ckpt_dir {ADMET_MODEL_CKPT_DIR} "
    f"--ckpt_files {model_files_str} "  
    f"--out_csv {admet_csv_path} "          
    f"--model_type custom "                
    f"--model_module src.admet_multi.model_gin_multi " 
    f"--model_class GINHead "              
    f"--hidden_dim 256 "                   
    f"--num_layers 6 "                     
    f"--dropout 0.2"                       
)

print(f"CMD: {cmd3}")
!{cmd3}

if not os.path.exists(admet_csv_path):
    print(f"❌ [3단계 오류] {admet_csv_path} 파일이 생성되지 않았습니다.")
else:
    print(f"\n[3단계 완료] ADMET 예측 결과 저장: {admet_csv_path}")
    print("\n--- ADMET 예측 결과 (CSV) ---")
    !head -n 5 {admet_csv_path}

[3단계 실행]... ADMET 예측을 시작합니다. (대조군 포함, 모델 파일 지정)
CMD: PYTHONPATH=/home/nudge/Project/drugai python -m src.admet_multi.predict_admet_gnn --smiles_file /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_with_controls.txt --ckpt_dir /home/nudge/Project/drugai/models --ckpt_files admet_hERG_optimized_complete.pt admet_CYP3A4_optimized_complete.pt admet_AqSolDB_optimized_complete.pt --out_csv /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/3_admet_results.csv --model_type custom --model_module src.admet_multi.model_gin_multi --model_class GINHead --hidden_dim 256 --num_layers 6 --dropout 0.2
[ok] wrote predictions: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/3_admet_results.csv (n=1024)

[3단계 완료] ADMET 예측 결과 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/3_admet_results.csv

--- ADMET 예측 결과 (CSV) ---
smiles,id,hERG_prob,hERG_flag,CYP3A4_prob,CYP3A4_flag,AqSolDB_pred
Oc1ccoc1,MOL_1,0.41897297,0,0.16134685,0,0

In [46]:
# [셀 3.5: ADMET 필터링 + 물성(LogP, Tanimoto, QED, SA) 사전 계산]

import os
import sys
import pandas as pd
from rdkit import Chem, DataStructs, RDConfig
from rdkit.Chem import AllChem, Descriptors, QED

# SA Score 모듈 로드 시도
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
try:
    import sascorer
    print("✅ SA_Score 모듈 로드 성공.")
except ImportError:
    sascorer = None
    print("⚠️ [경고] SA_Score 모듈을 찾을 수 없습니다. (점수 10.0 처리)")

print("[3.5단계 실행]... ADMET 필터링 및 물리화학적 속성 종합 계산.")

# --- ★★★ 설정 ★★★ ---
HERG_THRESHOLD = 0.5
CYP3A4_FLAG_PASS = 0 
SOLUBILITY_THRESHOLD = -4.0 
TAFAMIDIS_SMILES = "O=C(O)c1ccc2nc(oc2c1)c1cc(cc(c1)Cl)Cl"
# ---------------------

# 경로 설정
smiles_filtered_path = os.path.abspath(os.path.join(RESULTS_DIR, "SMILES_filtered.txt"))
props_csv_path = os.path.abspath(os.path.join(RESULTS_DIR, "3_5_filtered_with_props.csv"))

try:
    # 1. ADMET 결과 로드
    df_admet = pd.read_csv(admet_csv_path)
    df_admet = df_admet.rename(columns={"smiles": "SMILES", "id": "ID"}) 

    # 2. 필터링 로직
    control_ids = ["LIG_Aspirin", "LIG_T44", "LIG_Tafamidis"]
    condition_is_control = df_admet['ID'].isin(control_ids)
    
    condition_valid = (
        df_admet['hERG_prob'].notna() &
        df_admet['CYP3A4_flag'].notna() & 
        df_admet['AqSolDB_pred'].notna()
    )
    condition_herg = df_admet['hERG_prob'] < HERG_THRESHOLD
    condition_cyp = df_admet['CYP3A4_flag'] == CYP3A4_FLAG_PASS
    condition_sol = df_admet['AqSolDB_pred'] > SOLUBILITY_THRESHOLD

    df_passed = df_admet[
        (condition_is_control) | (condition_valid & condition_herg & condition_cyp & condition_sol)
    ].copy()

    print(f"  > 필터링 완료: {len(df_admet)}개 -> {len(df_passed)}개 통과")

    # =================================================================
    # ★ [추가됨] QED, SA Score, LogP, Tanimoto 계산
    # =================================================================
    print("  > 🧪 물성 지표(LogP, QED, SA, Tanimoto) 계산 중...")

    tafa_mol = Chem.MolFromSmiles(TAFAMIDIS_SMILES)
    tafa_fp = AllChem.GetMorganFingerprintAsBitVect(tafa_mol, 2, nBits=1024) if tafa_mol else None

    def calc_all_props(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if not mol: return None, None, None, None
            
            # 1. LogP
            logp = Descriptors.MolLogP(mol)
            
            # 2. QED (Drug-likeness)
            try: qed_val = QED.qed(mol)
            except: qed_val = 0.0

            # 3. SA Score (Synthetic Accessibility)
            if sascorer:
                try: sa_val = sascorer.calculateScore(mol)
                except: sa_val = 10.0
            else:
                sa_val = 10.0 # 모듈 없으면 최악의 점수 부여

            # 4. Tanimoto
            tanimoto = 0.0
            if tafa_fp:
                fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
                tanimoto = DataStructs.TanimotoSimilarity(tafa_fp, fp)
            
            return logp, qed_val, sa_val, tanimoto
        except:
            return None, None, None, None

    # 적용
    props = df_passed['SMILES'].apply(calc_all_props).apply(pd.Series)
    df_passed['LogP'] = props[0]
    df_passed['QED'] = props[1]       # ◀ 추가됨
    df_passed['SA_Score'] = props[2]  # ◀ 추가됨
    df_passed['Tanimoto_vs_Tafamidis'] = props[3]
    
    # =================================================================

    # 3. 저장
    df_passed[['SMILES', 'ID']].to_csv(smiles_filtered_path, sep=',', header=False, index=False)
    df_passed.to_csv(props_csv_path, index=False)
    
    print(f"\n✅ [3.5단계 완료]")
    print(f"   상세 정보(QED, SA 포함) 저장: {props_csv_path}")
    
    # 미리보기
    display(df_passed[['ID', 'LogP', 'QED', 'SA_Score']].head(5))

except Exception as e:
    print(f"❌ [3.5단계 오류]: {e}")

✅ SA_Score 모듈 로드 성공.
[3.5단계 실행]... ADMET 필터링 및 물리화학적 속성 종합 계산.
  > 필터링 완료: 1024개 -> 326개 통과
  > 🧪 물성 지표(LogP, QED, SA, Tanimoto) 계산 중...

✅ [3.5단계 완료]
   상세 정보(QED, SA 포함) 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/3_5_filtered_with_props.csv


,ID,LogP,QED,SA_Score
0,MOL_1,0.9852,0.508023,3.265848
5,MOL_8,1.3848,0.610604,1.054917
6,MOL_9,0.7855,0.585937,1.159218
8,MOL_11,0.2162,0.546234,3.503577
9,MOL_12,-0.3831,0.518766,3.665445


In [47]:
import torch
import gc

print("🧹 GPU 메모리 정리 중...")

# 파이썬 가비지 컬렉션
gc.collect()

# PyTorch 캐시 비우기
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
    # 현재 메모리 상태 출력
    t = torch.cuda.get_device_properties(0).total_memory
    r = torch.cuda.memory_reserved(0)
    a = torch.cuda.memory_allocated(0)
    f = r-a  # free inside reserved
    print(f"   > GPU 총 용량: {t/1024**3:.2f} GB")
    print(f"   > 사용 중: {a/1024**3:.2f} GB")
    print(f"   > 캐시됨(예약): {r/1024**3:.2f} GB")
    print("   ✅ 메모리 정리 완료.")
else:
    print("   ℹ️ GPU를 사용하지 않는 환경입니다.")

🧹 GPU 메모리 정리 중...
   > GPU 총 용량: 7.61 GB
   > 사용 중: 0.00 GB
   > 캐시됨(예약): 0.00 GB
   ✅ 메모리 정리 완료.


In [48]:
# [셀 4: 2단계 Binding Affinity 예측 (수정됨 - 숫자 변환 강제 & 디버깅)]

import pandas as pd
import os
import subprocess
import torch 

# 메모리 파편화 방지 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 이 셀은 'drugai' 커널에서 직접 실행됩니다.
print(f"[2단계 실행]... {len(RECEPTORS_CONFIG)}개 타겟에 대한 앙상블 친화도 예측을 시작합니다.")
print(f"   (원본 입력: {smiles_filtered_path})") 

# ------------------------------------------------------------------
# 입력 파일 포맷 보정
# ------------------------------------------------------------------
try:
    df_temp_input = pd.read_csv(smiles_filtered_path, header=None, names=['SMILES', 'ID'])
    temp_library_path = os.path.abspath(os.path.join(RESULTS_DIR, "temp_library_with_header.csv"))
    df_temp_input.to_csv(temp_library_path, index=False)
    print(f"   > 입력 파일 헤더 보정 완료: {temp_library_path}")
except Exception as e:
    print(f"❌ 입력 파일 읽기 실패: {e}")
    raise e

script_path = os.path.join(DRUGAI_DIR, "src", "binding_affinity", "screen.py")
script_dir = os.path.dirname(script_path)
script_name = os.path.basename(script_path)

all_affinity_files_created = True

# --- RECEPTORS_CONFIG 리스트를 순회하는 루프 ---
for target in RECEPTORS_CONFIG:
    target_name = target['name']
    target_pdb_path = target['pdb_path']
    target_center_coords = target['center']
    
    print(f"\n--- 타겟 처리 중: {target_name} ---")
    
    ensemble_dfs = []
    pocket_center_str = ",".join([str(coord) for coord in target_center_coords])
    
    # 최종 결과 파일 경로 (절대 경로)
    final_affinity_csv = os.path.abspath(os.path.join(RESULTS_DIR, f"2_affinity_results_{target_name}.csv"))

    # --- 앙상블 모델 루프 (3회 반복) ---
    for model_file in AFFINITY_ENSEMBLE_MODELS:
        model_path = os.path.join(AFFINITY_CKPT_DIR, model_file)
        seed_name = model_file.split('.')[0] 
        
        temp_csv_relative = os.path.join(RESULTS_DIR, f"temp_{target_name}_{seed_name}.csv")
        temp_csv = os.path.abspath(temp_csv_relative)
        
        cmd_ensemble = (
            f"cd {script_dir} && " 
            f"python {script_name} "
            f"--model_path {model_path} "
            f"--protein_path {target_pdb_path} "
            f"--library_path {temp_library_path} " 
            f"--output_path {temp_csv} " 
            f"--pocket_center \"{pocket_center_str}\" "
            f"--batch_size 16" 
        )
        
        try:
            result = subprocess.run(cmd_ensemble, shell=True, check=False, capture_output=True, text=True)
            
            if result.returncode != 0:
                print(f"  ❌ 실행 실패 ({seed_name})")
                err_lines = result.stderr.splitlines()
                short_err = "\n".join(err_lines[-20:]) if len(err_lines) > 20 else result.stderr
                print(f"  🔻 [에러 로그 요약]\n{short_err}")
                continue

            if os.path.exists(temp_csv):
                try:
                    df_temp = pd.read_csv(temp_csv)
                    
                    # [★핵심 수정] 강제로 숫자로 변환 (에러 발생 시 NaN 처리)
                    # 이렇게 하면 문자열이 섞여 있어도 TypeError가 나지 않음
                    if 'Predicted_pKd' in df_temp.columns:
                        df_temp['Predicted_pKd'] = pd.to_numeric(df_temp['Predicted_pKd'], errors='coerce')
                        
                        df_temp = df_temp.rename(columns={'Predicted_pKd': f'score_{seed_name}'})
                        ensemble_dfs.append(df_temp[['SMILES', f'score_{seed_name}']])
                    else:
                        print(f"  ⚠️ 경고: 결과 파일에 'Predicted_pKd' 컬럼이 없습니다.")
                        
                except Exception as e:
                    print(f"  ❌ 결과 CSV 파싱 실패 ({seed_name}): {e}")
                
                # 임시 파일 삭제
                os.remove(temp_csv)
            else:
                print(f"  ❌ 모델 {seed_name} 결과 파일 미생성.")
                print(f"  [stdout 로그 앞부분]\n{result.stdout[:500]}...") 
                
        except Exception as e:
            print(f"  ❌ 시스템 오류 ({seed_name}): {e}")

    # --- 앙상블 평균 계산 및 저장 ---
    if ensemble_dfs:
        df_final = ensemble_dfs[0]
        for df_next in ensemble_dfs[1:]:
            df_final = pd.merge(df_final, df_next, on='SMILES', how='outer')
            
        score_cols = [c for c in df_final.columns if c.startswith('score_')]
        
        # [추가 안전장치] 한번 더 숫자 변환 확인
        for col in score_cols:
             df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

        # 평균 계산 (NaN이 있어도 무시하고 계산됨)
        df_final['Predicted_pKd'] = df_final[score_cols].mean(axis=1)
        
        df_final[['SMILES', 'Predicted_pKd']].to_csv(final_affinity_csv, index=False)
        print(f"✅ {target_name} 앙상블({len(ensemble_dfs)}모델) 평균 결과 저장: {final_affinity_csv}")
    else:
        print(f"❌ [2단계 오류] {target_name}에 대한 예측 결과가 하나도 없습니다.")
        all_affinity_files_created = False

if all_affinity_files_created:
    print(f"\n[2단계 완료] 모든 타겟의 앙상블 예측 완료.")
else:
    print(f"\n[2단계 오류] 일부 타겟 실패. 로그 확인.")

[2단계 실행]... 2개 타겟에 대한 앙상블 친화도 예측을 시작합니다.
   (원본 입력: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/SMILES_filtered.txt)
   > 입력 파일 헤더 보정 완료: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/temp_library_with_header.csv

--- 타겟 처리 중: MT_V30M ---
✅ MT_V30M 앙상블(3모델) 평균 결과 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/2_affinity_results_MT_V30M.csv

--- 타겟 처리 중: TTR_WT ---
✅ TTR_WT 앙상블(3모델) 평균 결과 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/2_affinity_results_TTR_WT.csv

[2단계 완료] 모든 타겟의 앙상블 예측 완료.


In [49]:
# [셀 5: 4단계 결과 병합 및 핵심 지표(LogP, LLE, Tanimoto, QED, SA) 계산]

import os 
import json
import pandas as pd 
import glob 
import sys
from rdkit import Chem, DataStructs, RDConfig
from rdkit.Chem import AllChem, Descriptors, QED

# --- SA Score 모듈 로드 시도 ---
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
try:
    import sascorer
    print("✅ SA_Score 모듈 로드 성공.")
except ImportError:
    sascorer = None
    print("⚠️ [경고] SA_Score 모듈을 찾을 수 없습니다. (점수 10.0 처리)")

# 이 셀은 'drugai' 커널에서 직접 실행됩니다.
print("[4단계 실행]... 필터링된 결과 병합 및 모든 핵심 지표 계산.")

# 1. RESULTS_DIR 안전장치
if 'RESULTS_DIR' not in globals():
    dirs = sorted(glob.glob("./pipeline_results_*"), key=os.path.getmtime)
    if dirs:
        RESULTS_DIR = dirs[-1]
    else:
        raise NameError("RESULTS_DIR을 찾을 수 없습니다. [셀 1]을 먼저 실행하세요.")

final_csv_path = os.path.abspath(os.path.join(RESULTS_DIR, "FINAL_PIPELINE_RESULTS.csv"))

try:
    # --- 1. ADMET 결과 로드 ---
    if not os.path.exists(admet_csv_path):
        raise FileNotFoundError(f"ADMET 파일({admet_csv_path})을 찾을 수 없습니다.")
    
    df_admet = pd.read_csv(admet_csv_path)
    df_admet = df_admet.rename(columns={"smiles": "SMILES", "id": "ID"}) 

    # --- 2. 필터링된 SMILES+ID 로드 ---
    if not os.path.exists(smiles_filtered_path):
        raise FileNotFoundError(f"필터링된 SMILES 파일({smiles_filtered_path})을 찾을 수 없습니다.")
    
    df_smiles_base = pd.read_csv(
        smiles_filtered_path, 
        sep=r'[\s,]', 
        header=None, names=['SMILES', 'ID'],
        engine='python', on_bad_lines='skip'
    ).drop_duplicates(subset=['SMILES'])

    # --- 3. Affinity 결과 병합 ---
    df_final = df_smiles_base.copy() 
    
    for target in RECEPTORS_CONFIG:
        target_name = target['name']
        affinity_csv_path_target = os.path.abspath(os.path.join(RESULTS_DIR, f"2_affinity_results_{target_name}.csv"))

        if os.path.exists(affinity_csv_path_target):
            df_affinity = pd.read_csv(affinity_csv_path_target)
            target_col = 'Predicted_pKd' 
            new_col_name = f"pKd_ML_{target_name}"
            
            if target_col in df_affinity.columns:
                df_affinity = df_affinity.rename(columns={target_col: new_col_name})
                df_final = pd.merge(df_final, df_affinity[['SMILES', new_col_name]], on="SMILES", how="left")

    # --- 4. ADMET 결과 병합 ---
    merge_on_col = "ID" if "ID" in df_admet.columns else "SMILES"
    if "SMILES" in df_admet.columns and merge_on_col == "ID":
         df_admet_merged = df_admet.drop(columns=["SMILES"]) 
    else:
        df_admet_merged = df_admet
    
    df_final = pd.merge(df_final, df_admet_merged, on=merge_on_col, how="left")

    # ==================================================================
    # ★ [수정 핵심] LogP, Tanimoto, QED, SA Score 추가 계산
    # ==================================================================
    print("   > 🧪 추가 지표 계산 중 (LogP, QED, SA, Tanimoto, LLE)...")
    
    # Tafamidis 분자 준비
    tafa_mol = Chem.MolFromSmiles(TAFAMIDIS_SMILES)
    tafa_fp = AllChem.GetMorganFingerprintAsBitVect(tafa_mol, 2, nBits=1024) if tafa_mol else None
    
    def calc_properties(row):
        mol = Chem.MolFromSmiles(row['SMILES'])
        if not mol:
            return None, None, None, None
        
        # 1. LogP
        logp = Descriptors.MolLogP(mol)
        
        # 2. Tanimoto
        tanimoto = 0.0
        if tafa_fp:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
            tanimoto = DataStructs.TanimotoSimilarity(tafa_fp, fp)

        # 3. QED (Drug-likeness) ◀ [추가됨]
        try: qed_val = QED.qed(mol)
        except: qed_val = 0.0

        # 4. SA Score (Synthesizability) ◀ [추가됨]
        if sascorer:
            try: sa_val = sascorer.calculateScore(mol)
            except: sa_val = 10.0 # 계산 실패시 최악 점수
        else:
            sa_val = 10.0
            
        return logp, tanimoto, qed_val, sa_val

    # Apply 함수 적용 (결과 4개로 확장)
    props = df_final.apply(calc_properties, axis=1, result_type='expand')
    df_final['LogP'] = props[0]
    df_final['Tanimoto_vs_Tafamidis'] = props[1]
    df_final['QED'] = props[2]       # ◀ 컬럼 추가
    df_final['SA_Score'] = props[3]  # ◀ 컬럼 추가
    
    # 5. LLE 계산
    target_pkd_col = f"pKd_ML_{RECEPTORS_CONFIG[0]['name']}"
    if target_pkd_col in df_final.columns:
        df_final['LLE'] = df_final[target_pkd_col] - df_final['LogP']
    else:
        df_final['LLE'] = -999 

    # ==================================================================

    # --- 5. 정렬 ---
    control_ids = ["LIG_Aspirin", "LIG_T44", "LIG_Tafamidis"] 
    df_final['sort_priority'] = df_final['ID'].apply(lambda x: 0 if x in control_ids else 1)
    
    # 정렬 기준: 대조군 우선 -> LLE 높은순 (효율성) -> pKd 높은순
    df_final = df_final.sort_values(
        by=['sort_priority', 'LLE', target_pkd_col], 
        ascending=[True, False, False] 
    )
    df_final = df_final.drop(columns=['sort_priority'])

    # --- 최종 저장 ---
    df_final.to_csv(final_csv_path, index=False, float_format="%.3f")

    print(f"\n[4단계 완료] 최종 결과 저장: {final_csv_path}")
    
    # 🔻 확인용 출력 컬럼 확장 🔻
    print("\n--- Final Result (Top 5 by LLE) ---")
    cols_to_show = ['ID', target_pkd_col, 'LLE', 'QED', 'SA_Score', 'hERG_prob']
    
    # 존재하는 컬럼만 출력
    valid_cols = [c for c in cols_to_show if c in df_final.columns]
    display(df_final[valid_cols].head(5))

    # === 설정 파일 저장 ===
    output_config = {
        "DRUGAI_DIR": DRUGAI_DIR,
        "GENERATION_TARGET": GENERATION_TARGET,
        "RECEPTORS_CONFIG": RECEPTORS_CONFIG,
        "AFFINITY_ENSEMBLE_MODELS": AFFINITY_ENSEMBLE_MODELS,
        "FINAL_CSV_PATH": final_csv_path, 
        "RESULTS_DIR": os.path.abspath(RESULTS_DIR) 
    }
    
    config_save_path = os.path.join(RESULTS_DIR, "pipeline_output_config.json")
    with open(config_save_path, 'w') as f:
        json.dump(output_config, f, indent=4)
    print(f"✅ 설정 저장 완료: {config_save_path}")

except Exception as e:
    print(f"❌ [4단계 오류] : {e}")

✅ SA_Score 모듈 로드 성공.
[4단계 실행]... 필터링된 결과 병합 및 모든 핵심 지표 계산.
   > 🧪 추가 지표 계산 중 (LogP, QED, SA, Tanimoto, LLE)...

[4단계 완료] 최종 결과 저장: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/FINAL_PIPELINE_RESULTS.csv

--- Final Result (Top 5 by LLE) ---


,ID,pKd_ML_MT_V30M,LLE,QED,SA_Score,hERG_prob
325,LIG_Aspirin,4.186767,2.876667,0.550122,1.580040,0.283618
323,LIG_Tafamidis,5.277200,0.777400,0.756793,2.075533,0.701574
324,LIG_T44,5.039700,0.482400,0.388157,3.366456,0.847054
296,MOL_1255,5.879367,6.116167,0.632254,3.545440,0.383588
286,MOL_1210,5.889533,5.876533,0.598708,3.698673,0.202058


✅ 설정 저장 완료: ./pipeline_results_20251122_140342/pipeline_output_config.json


In [50]:
# [셀 6: 🏆 최종 유망 후보 선별 (pKd + QED/SA 필터링 적용)]

import pandas as pd
import json
import os

print("--- 🏆 최종 유망 후보 선별 및 2단계 전달 준비 ---")

# --- ★★★ [사용자 설정] 필터링 기준점 ★★★ ---
STRICT_PKD_THRESHOLD = 6.0   # 결합력 (기본)
QED_MIN_THRESHOLD = 0.6     # ◀ [추가] 약물성: 최소 0.6 이상 (관대하게)
SA_MAX_THRESHOLD = 6.0       # ◀ [추가] 합성난이도: 최대 6.0 이하 (실험 가능 마지노선)
# --------------------------------------------

# 1. 파일 로드
if os.path.exists(final_csv_path):
    df = pd.read_csv(final_csv_path)
else:
    raise FileNotFoundError(f"최종 결과 파일이 없습니다.")

# 2. Tafamidis 기준값 확인 (비교용)
try:
    tafa_row = df[df['ID'] == 'LIG_Tafamidis'].iloc[0]
    tafa_pkd = tafa_row['pKd_ML_MT_V30M']
    tafa_logp = tafa_row['LogP']
    tafa_lle = tafa_row['LLE']
    print(f"   > 💊 Tafamidis Spec: pKd={tafa_pkd:.2f}, LogP={tafa_logp:.2f}, LLE={tafa_lle:.2f}")
except:
    tafa_pkd, tafa_logp, tafa_lle = 6.6, 4.5, 2.1
    print(f"   > ⚠️ Tafamidis 데이터 없음. 기본값 사용.")

# 3. 필터링 적용
# (1) 대조군 제외
df_generated = df[~df['ID'].str.startswith('LIG_')].copy()

# (2) 조건 결합: pKd AND 선택성 AND QED AND SA
# 컬럼이 있는지 확인 후 필터링 (안전장치)
if 'QED' not in df_generated.columns or 'SA_Score' not in df_generated.columns:
    print("⚠️ [주의] QED 또는 SA_Score 컬럼이 없습니다. 해당 필터는 건너뜁니다.")
    # QED/SA가 없으면 0점/10점(최악)으로 채워서 에러 방지
    df_generated['QED'] = df_generated.get('QED', 0.0)
    df_generated['SA_Score'] = df_generated.get('SA_Score', 10.0)

candidates = df_generated[
    (df_generated['pKd_ML_MT_V30M'] > STRICT_PKD_THRESHOLD) &       # 결합력
    (df_generated['pKd_ML_MT_V30M'] >= df_generated['pKd_ML_TTR_WT']) & # 선택성
    (df_generated['QED'] >= QED_MIN_THRESHOLD) &                    # ◀ QED 조건
    (df_generated['SA_Score'] <= SA_MAX_THRESHOLD)                  # ◀ SA 조건
].copy()

# 정렬: LLE(효율성) 높은 순서대로
candidates = candidates.sort_values(by='LLE', ascending=False)

print(f"\n✅ 선별 결과 요약")
print(f"   - 기준: pKd > {STRICT_PKD_THRESHOLD}, QED >= {QED_MIN_THRESHOLD}, SA <= {SA_MAX_THRESHOLD}")
print(f"   - 전체 생성 분자: {len(df_generated)}개")
print(f"   - 🎯 통과한 후보(Hits): {len(candidates)}개")

if not candidates.empty:
    # 4. [핵심] 1단계 성적표 (Best Hit vs Tafamidis)
    best_hit = candidates.iloc[0]
    
    # 안전하게 값 가져오기
    best_qed = best_hit.get('QED', 0.0)
    best_sa = best_hit.get('SA_Score', 10.0)

    report_data = {
        "Metric": [
            "Binding Affinity (pKd)", 
            "Lipophilicity (LogP)", 
            "Efficiency (LLE)", 
            "Novelty (Tanimoto)",
            "Drug-likeness (QED)",  
            "Synthesizability (SA)" 
        ],
        "Tafamidis (Ref)": [
            f"{tafa_pkd:.2f}", f"{tafa_logp:.2f}", f"{tafa_lle:.2f}", 
            "1.00", "0.76", "2.10"
        ],
        "My Best Hit": [
            f"{best_hit['pKd_ML_MT_V30M']:.2f}", 
            f"{best_hit['LogP']:.2f}", 
            f"{best_hit['LLE']:.2f}", 
            f"{best_hit['Tanimoto_vs_Tafamidis']:.2f}",
            f"{best_qed:.2f}",      
            f"{best_sa:.2f}"        
        ],
        "Verdict": [
            "Stronger" if best_hit['pKd_ML_MT_V30M'] > tafa_pkd else "Comparable",
            "Better" if best_hit['LogP'] < tafa_logp else "Higher",
            "More Efficient" if best_hit['LLE'] > tafa_lle else "Lower",
            "New Scaffold" if best_hit['Tanimoto_vs_Tafamidis'] < 0.6 else "Similar",
            "Pass" if best_qed >= QED_MIN_THRESHOLD else "Fail",
            "Pass" if best_sa <= SA_MAX_THRESHOLD else "Fail"
        ]
    }
    print("\n📊 [Best Hit 상세 분석]")
    display(pd.DataFrame(report_data))

    # 5. 파일 저장
    best_hits_path = os.path.abspath(os.path.join(RESULTS_DIR, "BEST_CANDIDATES_FOR_EVOLUTION.csv"))
    candidates.to_csv(best_hits_path, index=False)
    print(f"   💾 [저장 완료] 2단계 입력 파일: {best_hits_path}")

    # Config 업데이트
    config_save_path = os.path.join(RESULTS_DIR, "pipeline_output_config.json")
    with open(config_save_path, 'r') as f:
        config_data = json.load(f)
    config_data['FINAL_CSV_PATH'] = best_hits_path 
    with open(config_save_path, 'w') as f:
        json.dump(config_data, f, indent=4)
    print(f"   🔗 [설정 업데이트] 노트북 2 연동 완료.")
    
    # 리스트 출력
    print("\n--- 🥇 Top 5 Candidates (Filtered & Sorted) ---")
    cols = [
        'ID', 'SMILES', 'pKd_ML_MT_V30M', 'LLE', 
        'QED', 'SA_Score', 
        'hERG_prob', 'AqSolDB_pred'
    ]
    valid_cols = [c for c in cols if c in candidates.columns]
    display(candidates[valid_cols].head(5))

else:
    print("\n❌ [경고] 조건에 맞는 후보가 하나도 없습니다!")
    print("   > 힌트: QED 기준을 0.60로 높이거나, SA 기준을 6.0으로 강화해보세요.")
    print("   > 우선 점수가 가장 높은 상위 5개를 보여드립니다 (참고용):")
    display(df_generated.sort_values(by='pKd_ML_MT_V30M', ascending=False).head(5))

--- 🏆 최종 유망 후보 선별 및 2단계 전달 준비 ---
   > 💊 Tafamidis Spec: pKd=5.28, LogP=4.50, LLE=0.78

✅ 선별 결과 요약
   - 기준: pKd > 6.0, QED >= 0.6, SA <= 6.0
   - 전체 생성 분자: 323개
   - 🎯 통과한 후보(Hits): 5개

📊 [Best Hit 상세 분석]


,Metric,Tafamidis (Ref),My Best Hit,Verdict
0,Binding Affinity (pKd),5.28,6.32,Stronger
1,Lipophilicity (LogP),4.50,1.12,Better
2,Efficiency (LLE),0.78,5.20,More Efficient
3,Novelty (Tanimoto),1.00,0.14,New Scaffold
4,Drug-likeness (QED),0.76,0.67,Pass
5,Synthesizability (SA),2.10,4.12,Pass


   💾 [저장 완료] 2단계 입력 파일: /home/nudge/Project/drugai/ipynb/pipeline_results_20251122_140342/BEST_CANDIDATES_FOR_EVOLUTION.csv
   🔗 [설정 업데이트] 노트북 2 연동 완료.

--- 🥇 Top 5 Candidates (Filtered & Sorted) ---


,ID,SMILES,pKd_ML_MT_V30M,LLE,QED,SA_Score,hERG_prob,AqSolDB_pred
21,MOL_1145,NC1=NC2=C(c3nc(=O)c4cc(O)cc(F)c4o3)C(O)=C=CC2=N1,6.320,5.196,0.672,4.125,0.138,-2.797
22,MOL_1133,NC1=NC2=C(c3nc(=O)c4cc(O)cc(F)c4o3)C=C=C(O)C2=N1,6.320,5.196,0.672,4.073,0.139,-2.843
70,MOL_809,O=C1N=C=C(O)C=C1c1ncc2cc(O)cc(F)c2n1,6.056,4.546,0.819,3.762,0.316,-2.193
147,MOL_1282,O=C1C=CC2=C(c3nc(=O)c4cc(O)cc(F)c4o3)C(O)=C=CC...,6.285,3.937,0.673,3.911,0.246,-3.669
234,MOL_1285,CC1=CC2=C(c3nc(=O)c4cc(O)cc(F)c4o3)C(O)=CCC2=N...,6.012,3.452,0.814,3.654,0.322,-3.321
